In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing._encoders import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RandomizedSearchCV

df = pd.read_csv("../data/processed/featured.csv")

# 1 Defining the X and Y
- Defining X that will be the features used to predict the target
- Defining y that is the predicted target

In [2]:
X = df.drop("Time_taken(min)",axis=1)
y = df["Time_taken(min)"]

# 2 Split 80/20
- using the method of splitting to split the data into 80% training data and 20% testing data

In [3]:
X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# 3 Identifying numerical and categorical features
- Identifying the numerical features
- Identifying the categorical features

In [4]:
numerical_features = X_train.select_dtypes(include=["Int64","Float64"]).columns
categorical_features = X_train.select_dtypes(exclude=["Int64","Float64"]).columns

# 4 Build the preprocessing pipeline

In [5]:
numerical_pipeline = Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("scaler",StandardScaler())
])

categorical_pipeline = Pipeline([
    ("impute",SimpleImputer(strategy="most_frequent")),
    ("scaler",OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ("num",numerical_pipeline,numerical_features),
    ("cat",categorical_pipeline,categorical_features)
])

# 5 Connect the preprocessor to the models

In [6]:
linear_regression_model = Pipeline([
    ("preprocessor",preprocessor),
    ("regressor",LinearRegression())
])

decision_tree_model = Pipeline([
    ("preprocessor",preprocessor),
    ("regressor",DecisionTreeRegressor())
])

random_forest_model = Pipeline([
    ("preprocessor",preprocessor),
    ("regressor",RandomForestRegressor())
])

gradient_boosting_model = Pipeline([
    ("preprocessor",preprocessor),
    ("regressor",GradientBoostingRegressor())
])

models = {
    "Linear Regression" : linear_regression_model,
    "Decision Tree Regressor" : decision_tree_model,
    "Random Forest" : random_forest_model,
    "Gradient Boosting" : gradient_boosting_model
}

# 6 Evaluate the models

In [7]:
testing_df = pd.DataFrame(columns=["MAE","MSE","RMSE","R2(%)"],index=["Linear Regression","Decision Tree Regressor","Random Forest","Gradient Boosting"])
training_df = pd.DataFrame(columns=["MAE","MSE","RMSE","R2(%)"],index=["Linear Regression","Decision Tree Regressor","Random Forest","Gradient Boosting"])
x_validation_df = pd.DataFrame(columns=["Mean MAE","MAE_STD","Mean MSE","MSE_STD","Mean RMSE","RMSE_STD","Mean R2","R2_STD"],index=["Linear Regression","Decision Tree Regressor","Random Forest","Gradient Boosting"])

scoring = {
    "MAE": "neg_mean_absolute_error",
    "MSE": "neg_mean_squared_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2"
}

for name, model in models.items():
    for metric, scoring_name in scoring.items():
        scores = cross_val_score(
            model, X_train, y_train,
            cv=5,
            scoring=scoring_name
        )

        if scoring_name.startswith("neg_"):
            scores = -scores

        x_validation_df.loc[name, f"Mean {metric}"] = scores.mean()
        x_validation_df.loc[name, f"{metric}_STD"] = scores.std()

for name,model in models.items():
    model.fit(X_train,y_train)

    predictions = model.predict(X_test)
    test_evaluations = [
        mean_absolute_error(y_test,predictions),
        mean_squared_error(y_test,predictions),
        np.sqrt(mean_squared_error(y_test,predictions)),
        round(r2_score(y_test,predictions)*100,2)
    ]
    testing_df.loc[name] = test_evaluations

    predictions = model.predict(X_train)
    train_evaluations = [
        mean_absolute_error(y_train,predictions),
        mean_squared_error(y_train,predictions),
        np.sqrt(mean_squared_error(y_train,predictions)),
        round(r2_score(y_train,predictions)*100,2)
    ]
    training_df.loc[name] = train_evaluations

print("Cross Validation :")
print(x_validation_df)
print("="*40)
print("Testing Results")
print(testing_df.sort_values(by="R2(%)",ascending=False))
print("="*40)
print("Training Results")
print(training_df.sort_values(by="R2(%)",ascending=False))
print("\n")

Cross Validation :
                         Mean MAE   MAE_STD   Mean MSE   MSE_STD Mean RMSE  \
Linear Regression        4.743862  0.034448  35.492309  0.507344   5.95739   
Decision Tree Regressor  4.126643  0.024358  29.309178  0.237605  5.403918   
Random Forest            3.160135  0.022783   15.78004  0.179001  3.974421   
Gradient Boosting         3.55566  0.013996  19.731559  0.265819  4.441922   

                         RMSE_STD   Mean R2    R2_STD  
Linear Regression        0.042624  0.594554  0.005233  
Decision Tree Regressor  0.031584  0.664204  0.007757  
Random Forest            0.024964  0.819753  0.002931  
Gradient Boosting        0.029814  0.774574  0.004174  
Testing Results
                              MAE        MSE      RMSE  R2(%)
Random Forest            3.152464  15.747496  3.968311  81.96
Gradient Boosting        3.578363  20.041777  4.476804  77.04
Decision Tree Regressor  4.094927  28.903773  5.376223  66.89
Linear Regression        4.776158  35.994743  

In [ ]:
param_grid = {
    "regressor__n_estimators": [100, 200, 300, 500],
    "regressor__max_depth": [None, 10, 20, 30],
    "regressor__min_samples_split": [2, 5, 10],
    "regressor__min_samples_leaf": [1, 2, 4],
    "regressor__max_features": ["sqrt", "log2", None]
}

random_search = RandomizedSearchCV(
    estimator=random_forest_model,
    param_distributions=param_grid,
    n_iter=20,
    cv=5,
    scoring="neg_mean_absolute_error",
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train,y_train)

print(random_search.best_params_)

print(-random_search.best_score_)

best_rf = random_search.best_estimator_

y_pred = best_rf.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R2:", r2)